In [1]:
# Set knit directory
setwd(here::here('./01_euc_saligna/'))
knitr::opts_knit$set(root.dir = setwd(here::here('./01_euc_saligna/')))
getwd()

[1] "/home/ecamo19/Documents/projects/eucalyptus_heat_responses/01_euc_saligna"

In [2]:
library(scales)
library(ggplot2)
library(HIEdroughtbox)

# For data manipulation
library(here)
library(dplyr)
library(purrr)
library(tidyr)
library(tibble)
library(stringr)
library(lubridate)

here() starts at /home/ecamo19/Documents/projects/eucalyptus_heat_responses


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘purrr’


The following object is masked from ‘package:scales’:

    discard



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




In [ ]:
string_colors <- c(string_1 = "#eec000",
                   string_2 = "#cf544c",
                   string_3 = "#0175c3",
                   string_4 = "#878687",
                   
                   string_5 =  "#000000",
                   string_6 =  "#008b28",
                   string_7 =  "#8600b6",
                   string_8 =  "#00a0ab")

# 1. Load Droughtbox data

In [ ]:
euc_saligna_raw_data <- read_hie_droughtbox_data_folder("./raw_euc_saligna_apr_droughtbox_data")

In [ ]:
names(euc_saligna_raw_data)

# 2. Clean Droughtbox data

Remember to remove the `eval = FALSE` from each chunk

## Plot raw droughtbox data

In [ ]:
map(euc_saligna_raw_data[!grepl("no-vpd-control", 
                               names(euc_saligna_raw_data))], 
    plot_strains_weights)

## __Data with VPD controlled__ 

### File 1: 25c

In [ ]:
euc_saligna_vpd_controlled_25c_filtered <- 
    
    # Select the data to be clean
    euc_saligna_raw_data["euc_saligna_apr_25c.dat"] %>% 
    
    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
    
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%
  
    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "01:05:00",
                           to_end_time = "12:40:00") %>% 
    
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 

    # Step done to avoid breaking the time sequence
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_saligna_vpd_controlled_25c_filtered, show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_saligna_vpd_controlled_25c_reshaped <- reshape_droughtbox_data(euc_saligna_vpd_controlled_25c_filtered)

In [ ]:
# Check data
any(euc_saligna_vpd_controlled_25c_reshaped$temperature_measured < 25 |
    euc_saligna_vpd_controlled_25c_reshaped$temperature_measured > 25)

#### Clean reshaped data

In [ ]:
euc_saligna_vpd_controlled_25c_reshaped_cleaned <- 
  
  euc_saligna_vpd_controlled_25c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 25c
        #filter( temperature_measured <= 25)  %>% 
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>% 
        #filter(!c(string_number == "string_6")) %>% 
    
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 14)) %>%
        filter(!c(string_number == "string_2" & string_weight_grams < 14)) %>%
        filter(!c(string_number == "string_3" & string_weight_grams < 17.8)) %>%
        filter(!c(string_number == "string_4" & string_weight_grams < 14)) %>%

        filter(!c(string_number == "string_5" & string_weight_grams < 14)) %>%
        filter(!c(string_number == "string_6" & string_weight_grams < 14)) %>%
        filter(!c(string_number == "string_7" & string_weight_grams < 14)) %>%
        filter(!c(string_number == "string_8" & string_weight_grams < 14)) %>%

        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
          
        filter(!c(string_number == "string_3" & string_weight_grams < 18.5 & time_seconds < 4000)) %>%
        filter(!c(string_number == "string_4" & string_weight_grams < 18.5 & time_seconds < 14000)) %>%
  
        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Visualize data before summarizing
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_saligna_vpd_controlled_25c_reshaped_cleaned, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour = factor(string_number))) +

    geom_point()+

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 2: 30c

In [ ]:
euc_saligna_vpd_controlled_30c_filtered <-

    # Select the data to be clean
    euc_saligna_raw_data["euc_saligna_apr_30c.dat"] %>%

    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%
  
    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_date = "2025-04-15",
                           to_end_date =  "2025-04-15",
                           from_start_time = "00:20:00",
                           to_end_time = "12:50:00") %>% 
    
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 

    # Step done to avoid breaking the time sequence
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_saligna_vpd_controlled_30c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_saligna_vpd_controlled_30c_reshaped <- 
    reshape_droughtbox_data(euc_saligna_vpd_controlled_30c_filtered)

In [ ]:
# Check data
any(euc_saligna_vpd_controlled_30c_reshaped$temperature_measured < 30 |
    euc_saligna_vpd_controlled_30c_reshaped$temperature_measured > 30)

#### Clean reshaped data

In [ ]:
euc_saligna_vpd_controlled_30c_reshaped_cleaned  <- 
  
    euc_saligna_vpd_controlled_30c_reshaped %>%   
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 30c
        #filter( temperature_measured <= 30c)  %>% 
        
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 14)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 14)) %>% 
        
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        #Visualize data before summarizing
        summarise(across(tc_avg_deg_c_avg:string_weight_grams,
                         \(x) median(x, na.rm = TRUE))) %>%
         
        arrange(date_time)

In [ ]:
ggplot(data = euc_saligna_vpd_controlled_30c_reshaped_cleaned, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour =factor(string_number))) +

    geom_point()+

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 3: 35c

In [ ]:
euc_saligna_vpd_controlled_35c_filtered <-

    # Select the data to be clean
    euc_saligna_raw_data["euc_saligna_apr_35c.dat"] %>%  

    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "12:00:00",
                           to_end_time = "18:30:00") %>%
  
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 
    
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_saligna_vpd_controlled_35c_filtered, show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_saligna_vpd_controlled_35c_reshaped <- reshape_droughtbox_data(euc_saligna_vpd_controlled_35c_filtered)

In [ ]:
# Check data
any(euc_saligna_vpd_controlled_35c_reshaped$temperature_measured < 35 |
    euc_saligna_vpd_controlled_35c_reshaped$temperature_measured > 35)

#### Clean reshaped data

In [ ]:
euc_saligna_vpd_controlled_35c_reshaped_cleaned <- 
  
    euc_saligna_vpd_controlled_35c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 35c
        #filter( temperature_measured <= 35)  %>% 
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>% 
        #filter(!c(string_number == "string_6")) %>% 
    
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 10)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 10)) %>% 

        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
  
               
        filter(!c(string_number == "string_3" & string_weight_grams < 14)) %>%
        
        # Removing data bumps 
        filter(!c(string_number == "string_7" & time_seconds > 5000 & time_seconds <= 6000)) %>%
        filter(!c(string_number == "string_7" & time_seconds > 9000 & time_seconds < 11000)) %>%
        filter(!c(string_number == "string_7" & time_seconds > 13000 & time_seconds < 16000)) %>%
        filter(!c(string_number == "string_7" & time_seconds >= 18000 & time_seconds < 20000)) %>%


        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams,
                         \(x) median(x, na.rm = TRUE))) %>%
        # 
        arrange(date_time)

In [ ]:
ggplot(data = euc_saligna_vpd_controlled_35c_reshaped_cleaned, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour = factor(string_number))) +

    geom_point()+

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 4: 40c

In [ ]:
euc_saligna_vpd_controlled_40c_filtered <-

    # Select the data to be clean
    euc_saligna_raw_data["euc_saligna_apr_40c.dat"] %>% 

    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "19:00:00",
                           to_end_time = "23:59:59") %>%
  
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 
  
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_saligna_vpd_controlled_40c_filtered, show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_saligna_vpd_controlled_40c_reshaped <- reshape_droughtbox_data(euc_saligna_vpd_controlled_40c_filtered)

In [ ]:
# Check data
any(euc_saligna_vpd_controlled_40c_reshaped$temperature_measured < 40 |
    euc_saligna_vpd_controlled_40c_reshaped$temperature_measured >  40)

#### Clean reshaped data

In [ ]:
euc_saligna_vpd_controlled_40c_reshaped_cleaned <- 
  
    euc_saligna_vpd_controlled_40c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 40c
        #filter( temperature_measured <= 40) %>%
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>%
        #filter(!c(string_number == "string_6")) %>% 

        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 18)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 12)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 12)) %>% 
        
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
  
        # Removing data bumps 
        filter(!c(string_number == "string_3" & string_weight_grams < 20 & time_seconds > 1000 & time_seconds <= 6000)) %>%

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        #Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams,
                        \(x) median(x, na.rm = TRUE))) %>%
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_saligna_vpd_controlled_40c_reshaped_cleaned, aes(x = time_seconds,
                                                              y = string_weight_grams,
                                                              colour = factor(string_number))) +

    geom_point() +

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 5: 45c

In [ ]:
euc_saligna_vpd_controlled_45c_filtered <-

    # Select the data to be clean
    euc_saligna_raw_data["euc_saligna_apr_45c.dat"] %>% 

    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "13:40:00",
                           to_end_time = "17:40:00") %>%
  
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 
    
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_saligna_vpd_controlled_45c_filtered, show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_saligna_vpd_controlled_45c_reshaped <- reshape_droughtbox_data(euc_saligna_vpd_controlled_45c_filtered)

In [ ]:
# Check data
any(euc_saligna_vpd_controlled_45c_reshaped$temperature_measured < 45 |
    euc_saligna_vpd_controlled_45c_reshaped$temperature_measured > 45)

#### Clean reshaped data

In [ ]:
euc_saligna_vpd_controlled_45c_reshaped_cleaned <- 
  
    euc_saligna_vpd_controlled_45c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 45c
        #filter( temperature_measured <= 45)  %>% 
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>%
        #filter(!c(string_number == "string_6")) %>% 
    
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 12)) %>%
        filter(!c(string_number == "string_2" & string_weight_grams < 12)) %>%
        filter(!c(string_number == "string_3" & string_weight_grams < 12)) %>%
        filter(!c(string_number == "string_4" & string_weight_grams < 12)) %>%

        filter(!c(string_number == "string_5" & string_weight_grams < 12)) %>%
        filter(!c(string_number == "string_6" & string_weight_grams < 12)) %>%
        filter(!c(string_number == "string_7" & string_weight_grams < 12)) %>%
        filter(!c(string_number == "string_8" & string_weight_grams < 12)) %>%

        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minute"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
  
        # Removing data bumps 
        filter(!c(string_number == "string_7" & time_seconds > 2000 & time_seconds < 3000)) %>%
        filter(!c(string_number == "string_7" & time_seconds > 6500 & time_seconds < 8000)) %>%
        filter(!c(string_number == "string_7" & time_seconds > 10900 & time_seconds < 12700)) %>%

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_saligna_vpd_controlled_45c_reshaped_cleaned, aes(x = time_seconds,
                                                              y = string_weight_grams,
                                                              colour = factor(string_number))) +

    geom_point()+

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 6: 50c

In [ ]:
euc_saligna_vpd_controlled_50c_filtered <- 
    
    # Select the data to be clean
    euc_saligna_raw_data["euc_saligna_apr_50c.dat"] %>% 
    
    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "18:20:00",
                           to_end_time = "21:10:00") %>% 
  
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>%   
  
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_saligna_vpd_controlled_50c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_saligna_vpd_controlled_50c_reshaped <- reshape_droughtbox_data(euc_saligna_vpd_controlled_50c_filtered)

In [ ]:
# Check data
any(euc_saligna_vpd_controlled_50c_reshaped$temperature_measured < 50| 
    euc_saligna_vpd_controlled_50c_reshaped$temperature_measured > 50)

#### Clean reshaped data

In [ ]:
euc_saligna_vpd_controlled_50c_reshaped_cleaned <- 
  
    euc_saligna_vpd_controlled_50c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 50c
        #filter( temperature_measured <= 50)  %>%
        
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 14)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 14)) %>% 

        # Filter observations if required
        #filter(!c(string_number == "string_7" & string_weight_grams < 16)) %>%
        #filter(!c(string_number == "string_8" & string_weight_grams < 16))
 
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        #Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams,
                        \(x) median(x, na.rm = TRUE))) %>%
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_saligna_vpd_controlled_50c_reshaped_cleaned, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour = factor(string_number))) +

    geom_point()+

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 7: 55c

In [ ]:
euc_saligna_vpd_controlled_55c_filtered <- 
    
    # Select the data to be clean
    euc_saligna_raw_data["euc_saligna_apr_55c.dat"] %>% 
    
    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "14:05:00",
                           to_end_time = "17:05:00") %>% 
    
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 
  
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_saligna_vpd_controlled_55c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_saligna_vpd_controlled_55c_reshaped <- reshape_droughtbox_data(euc_saligna_vpd_controlled_55c_filtered)

In [ ]:
# Check data
any(euc_saligna_vpd_controlled_55c_reshaped$temperature_measured < 55| 
    euc_saligna_vpd_controlled_55c_reshaped$temperature_measured > 55)

#### Clean reshaped data

In [ ]:
euc_saligna_vpd_controlled_55c_reshaped_cleaned <- 
  
    euc_saligna_vpd_controlled_55c_reshaped %>%  
      
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 55c
        #filter(temperature_measured >= 55)  %>% 
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>%
        filter(!c(string_number == "string_7")) %>% 
    
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 14)) %>%
        filter(!c(string_number == "string_2" & string_weight_grams < 14)) %>%
        filter(!c(string_number == "string_3" & string_weight_grams < 18)) %>%
        filter(!c(string_number == "string_4" & string_weight_grams < 20)) %>%

        filter(!c(string_number == "string_5" & string_weight_grams < 14)) %>%
        filter(!c(string_number == "string_6" & string_weight_grams < 14)) %>%
        filter(!c(string_number == "string_7" & string_weight_grams < 14)) %>%
        filter(!c(string_number == "string_8" & string_weight_grams < 14)) %>%

        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                            unit = "second")) %>% 

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
          
        arrange(date_time)

In [ ]:
ggplot(data = euc_saligna_vpd_controlled_55c_reshaped_cleaned, aes(x = time_seconds,
                                                              y = string_weight_grams,
                                                              colour = factor(string_number))) +

    geom_point() +

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### Merge cleaned dataframes

In [ ]:

euc_saligna_vpd_controlled_merged_cleaned_data <-
 
        list(
            euc_saligna_vpd_controlled_25c_reshaped_cleaned,
            euc_saligna_vpd_controlled_30c_reshaped_cleaned,
            euc_saligna_vpd_controlled_35c_reshaped_cleaned,
            euc_saligna_vpd_controlled_40c_reshaped_cleaned,
            euc_saligna_vpd_controlled_45c_reshaped_cleaned,
            euc_saligna_vpd_controlled_50c_reshaped_cleaned,
            euc_saligna_vpd_controlled_55c_reshaped_cleaned) %>%    
            
    reduce(rbind) %>% 
    add_column(vpd_control = factor("yes")) %>%
    arrange(string_number)

#### Plot Merged data

__Make sure that values start at 0__

In [ ]:
ggplot(data = euc_saligna_vpd_controlled_merged_cleaned_data, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour = factor(string_number))) +
  
    geom_point() +
    
    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors  
    scale_color_manual(values = string_colors) +
    
    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +
    
    scale_x_continuous(breaks = scales::pretty_breaks(n = 10))

In [ ]:
glimpse(euc_saligna_vpd_controlled_merged_cleaned_data)

## __Data with VPD _not_ controlled__ 

### Plot raw data

In [ ]:
map(euc_saligna_raw_data[grepl("no-vpd-control", names(euc_saligna_raw_data))], plot_strains_weights)

### File 2: 30c

In [ ]:
euc_saligna_no_vpd_control_30c_filtered <-

    # Select the data to be clean
    euc_saligna_raw_data["euc_saligna_apr_no-vpd-control_30c.dat"] %>%

    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%
  
    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_date = "2025-04-17",
                           to_end_date =  "2025-04-17",
                           from_start_time = "00:00:00",
                           to_end_time = "13:00:00") %>% 

    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 

    # Step done to avoid breaking the time sequence
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_saligna_no_vpd_control_30c_filtered, show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_saligna_no_vpd_control_30c_reshaped  <- 
    reshape_droughtbox_data(euc_saligna_no_vpd_control_30c_filtered)

In [ ]:
# Check data
any(euc_saligna_no_vpd_control_30c_reshaped$temperature_measured < 30 |
    euc_saligna_no_vpd_control_30c_reshaped$temperature_measured > 30)

#### Clean reshaped data

In [ ]:
euc_saligna_no_vpd_control_30c_reshaped_cleaned  <- 
  
    euc_saligna_no_vpd_control_30c_reshaped %>%   
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 30c
        #filter( temperature_measured <= 30c)  %>% 
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>%
        #filter(!c(string_number == "string_6")) %>% 
        
        ## Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 13)) %>%
        filter(!c(string_number == "string_2" & string_weight_grams < 13)) %>%
        filter(!c(string_number == "string_3" & string_weight_grams < 13)) %>%
        filter(!c(string_number == "string_4" & string_weight_grams < 13)) %>%

        filter(!c(string_number == "string_5" & string_weight_grams < 13)) %>%
        filter(!c(string_number == "string_6" & string_weight_grams < 13)) %>%
        filter(!c(string_number == "string_7" & string_weight_grams < 13)) %>%
        filter(!c(string_number == "string_8" & string_weight_grams < 13)) %>%

        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minute"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
  
        # Removing data bumps 
        filter(!c(string_number == "string_3" & time_seconds > 2000 & time_seconds < 20000 & string_weight_grams < 16 )) %>%
  
        filter(!c(string_number == "string_4" & time_seconds > 10000 & time_seconds < 12000 & string_weight_grams < 18 )) %>%
        #filter(!c(string_number == "string_7" & time_seconds > 6500 & time_seconds < 8000)) %>%
        #filter(!c(string_number == "string_7" & time_seconds > 10900 & time_seconds < 12700)) %>%

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Visualize data before summarizing
        summarise(across(tc_avg_deg_c_avg:string_weight_grams,
                         \(x) median(x, na.rm = TRUE))) %>%
         
        arrange(date_time)

In [ ]:
ggplot(data = euc_saligna_no_vpd_control_30c_reshaped_cleaned, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour =factor(string_number))) +

    geom_point() +

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20)) +
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 3: 35c

In [ ]:
euc_saligna_no_vpd_control_35c_filtered <-

    # Select the data to be clean
    euc_saligna_raw_data["euc_saligna_apr_no-vpd-control_35c.dat"] %>%

    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "13:50:00",
                           to_end_time = "19:40:00") %>%
    
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 
    
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_saligna_no_vpd_control_35c_filtered, show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_saligna_no_vpd_control_35c_reshaped <-
    reshape_droughtbox_data(euc_saligna_no_vpd_control_35c_filtered)

In [ ]:
# Check data
any(euc_saligna_vpd_controlled_35c_reshaped$temperature_measured < 35 |
    euc_saligna_vpd_controlled_35c_reshaped$temperature_measured > 35)

#### Clean reshaped data

In [ ]:
euc_saligna_no_vpd_control_35c_reshaped_cleaned <- 
  
    euc_saligna_no_vpd_control_35c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 35c
        #filter( temperature_measured <= 35)  %>% 
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>% 
        #filter(!c(string_number == "string_6")) %>% 
    
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 14)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 14)) %>% 
        
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minute"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
  
        filter(!c(string_number == "string_3" & string_weight_grams < 15.5 & time_seconds > 4000 & time_seconds < 11000)) %>%
        
        filter(!c(string_number == "string_4" & string_weight_grams < 17.5 & time_seconds > 5000 & time_seconds < 6000)) %>% 
  
        filter(!c(string_number == "string_4" & string_weight_grams < 14.5 & time_seconds > 16000 & time_seconds < 17000)) %>% 
        

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_saligna_no_vpd_control_35c_reshaped_cleaned, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour = factor(string_number))) +

    geom_point() +

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 4: 40c

In [ ]:
euc_saligna_no_vpd_control_40c_filtered <-

    # Select the data to be clean
    euc_saligna_raw_data["euc_saligna_apr_no-vpd-control_40c.dat"] %>%

    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "18:30:00",
                           to_end_time = "23:20:00") %>%
    
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 
  
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_saligna_no_vpd_control_40c_filtered, 
                    show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_saligna_no_vpd_control_40c_reshaped <-
    reshape_droughtbox_data(euc_saligna_no_vpd_control_40c_filtered)

In [ ]:
# Check data
any(euc_saligna_no_vpd_control_40c_reshaped$temperature_measured < 40 |
    euc_saligna_no_vpd_control_40c_reshaped$temperature_measured > 40)

#### Clean reshaped data

In [ ]:
euc_saligna_no_vpd_control_40c_reshaped_cleaned <- 
  
    euc_saligna_no_vpd_control_40c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 40c
        #filter( temperature_measured <= 40) %>%
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>%
        #filter(!c(string_number == "string_6")) %>% 

        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 16)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 13)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 15)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 13)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 12)) %>% 
        
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minute"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
  
        filter(!c(string_number == "string_3" & string_weight_grams < 20 & time_seconds > 2000 & time_seconds < 8000)) %>%
  
        filter(!c(string_number == "string_4" & string_weight_grams < 20 & time_seconds > 2000 & time_seconds < 8000)) %>%

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_saligna_no_vpd_control_40c_reshaped_cleaned, aes(x = time_seconds,
                                                              y = string_weight_grams,
                                                              colour = factor(string_number))) +

    geom_point() +

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 5: 45c

In [ ]:
euc_saligna_no_vpd_control_45c_filtered <-

    # Select the data to be clean
    euc_saligna_raw_data["euc_saligna_apr_no-vpd-control_45c.dat"] %>%

    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_date = "2025-04-17",
                           to_end_date =  "2025-04-18",
                           from_start_time = "20:20:00",
                           to_end_time = "00:10:00") %>% 
  
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 
    
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_saligna_no_vpd_control_45c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_saligna_no_vpd_control_45c_reshaped <- 
    reshape_droughtbox_data(euc_saligna_no_vpd_control_45c_filtered)

In [ ]:
# Check data
any(euc_saligna_no_vpd_control_45c_reshaped$temperature_measured < 45 |
    euc_saligna_no_vpd_control_45c_reshaped$temperature_measured > 45)

#### Clean reshaped data

In [ ]:
euc_saligna_no_vpd_control_45c_reshaped_cleaned <- 
  
    euc_saligna_no_vpd_control_45c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 45c
        #filter( temperature_measured <= 45) %>% 
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>%
        #filter(!c(string_number == "string_6")) %>% 
    
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 10)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 10)) %>% 
        
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
        
        filter(!c(string_number == "string_1" & string_weight_grams < 17 & time_seconds > 3000 & time_seconds < 4000)) %>%
  
  
        filter(!c(string_number == "string_3" & string_weight_grams < 20 & time_seconds >= 3000 & time_seconds <= 6000)) %>%
        
        filter(!c(string_number == "string_3" & string_weight_grams < 13 & time_seconds > 12000 & time_seconds < 13000)) %>%


        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_saligna_no_vpd_control_45c_reshaped_cleaned, aes(x = time_seconds,
                                                              y = string_weight_grams,
                                                              colour = factor(string_number))) +

    geom_point()+

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 6: 50c

In [ ]:
euc_saligna_no_vpd_control_50c_filtered <- 
    
    # Select the data to be clean
    euc_saligna_raw_data["euc_saligna_apr_no-vpd-control_50c.dat"] %>% 
    
    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "14:10:00",
                           to_end_time = "16:10:00") %>% 
    
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>%   
  
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_saligna_no_vpd_control_50c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_saligna_no_vpd_control_50c_reshaped <- 
    reshape_droughtbox_data(euc_saligna_no_vpd_control_50c_filtered)

In [ ]:
# Check data
any(euc_saligna_no_vpd_control_50c_reshaped$temperature_measured < 50| 
    euc_saligna_no_vpd_control_50c_reshaped$temperature_measured > 50)

In [ ]:
plot(euc_saligna_no_vpd_control_50c_reshaped$temperature_measured, 
     euc_saligna_no_vpd_control_50c_reshaped$tc_avg_deg_c_avg)

#### Clean reshaped data

In [ ]:
euc_saligna_no_vpd_control_50c_reshaped_cleaned  <- 
  
    euc_saligna_no_vpd_control_50c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 50c
        filter(temperature_measured == 50)  %>%
        
        # Filter strings if required
    
        #filter(!c(string_number == "string_5")) %>%
        #filter(!c(string_number == "string_6")) %>% 

        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 10)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 14)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 10)) %>% 
        
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
        
        filter(!c(string_number == "string_3" & string_weight_grams > 24 & time_seconds >= 500 & time_seconds <= 1000)) %>%
  
        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_saligna_no_vpd_control_50c_reshaped_cleaned, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour = factor(string_number))) +

    geom_point() +

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 7: 55c

In [ ]:
euc_saligna_no_vpd_control_55c_filtered <- 
    
    # Select the data to be clean
    euc_saligna_raw_data["euc_saligna_apr_no-vpd-control_55c.dat"] %>% 
    
    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "16:50:00",
                           to_end_time = "18:20:00") %>%
  
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 
  
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_saligna_no_vpd_control_55c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_saligna_no_vpd_control_55c_reshaped <-
    reshape_droughtbox_data(euc_saligna_no_vpd_control_55c_filtered)

In [ ]:
# Check data
any(euc_saligna_no_vpd_control_55c_reshaped$temperature_measured < 55| 
    euc_saligna_no_vpd_control_55c_reshaped$temperature_measured > 55)

#### Clean reshaped data

In [ ]:
euc_saligna_no_vpd_control_55c_reshaped_cleaned <- 
  
    euc_saligna_no_vpd_control_55c_reshaped %>%  
      
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 55c
        #filter(temperature_measured >= 55)  %>% 
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>%
        #filter(!c(string_number == "string_6")) %>% 
    
        ## Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 12)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 13)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 12)) %>% 
        
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                            unit = "second")) %>% 
  
      filter(!c(string_number == "string_6" & string_weight_grams < 16 & time_seconds >= 0 & time_seconds < 1000)) %>%
  
        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
          
        arrange(date_time)

In [ ]:
ggplot(data = euc_saligna_no_vpd_control_55c_reshaped_cleaned, aes(x = time_seconds,
                                                              y = string_weight_grams,
                                                              colour = factor(string_number))) +

    geom_point() +

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20)) +
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### Merge cleaned dataframes

In [ ]:

euc_saligna_no_vpd_control_merged_cleaned_data <-
 
        list(
            euc_saligna_no_vpd_control_30c_reshaped_cleaned,
            euc_saligna_no_vpd_control_35c_reshaped_cleaned,
            euc_saligna_no_vpd_control_40c_reshaped_cleaned,
            euc_saligna_no_vpd_control_45c_reshaped_cleaned,
            euc_saligna_no_vpd_control_50c_reshaped_cleaned,
            euc_saligna_no_vpd_control_55c_reshaped_cleaned) %>%    
            
    reduce(rbind) %>% 
    add_column(vpd_control = factor("no")) %>%
    arrange(string_number)

#### Plot Merged data

__Make sure that values start at 0__

In [ ]:
ggplot(data = euc_saligna_no_vpd_control_merged_cleaned_data, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour = factor(string_number))) +
  
    geom_point()+
    
    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors  
    scale_color_manual(values = string_colors) +
    
    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +
    
    scale_x_continuous(breaks = scales::pretty_breaks(n = 10))

In [ ]:
glimpse(euc_saligna_no_vpd_control_merged_cleaned_data)

## Merge datasets

In [ ]:
euc_saligna_droughtbox_data_cleaned <- 
            list(
              euc_saligna_no_vpd_control_merged_cleaned_data,
              euc_saligna_vpd_controlled_merged_cleaned_data
              ) %>%    
            
    reduce(rbind) %>%
    arrange(vpd_control, temperature_measured) %>%

    # Reorganize columns  
    select(date_time, string_number, temperature_measured, vpd_control, everything())

# 3. Add dry weights to cleaned dataframe

## Load dry weights data  

In [ ]:
euc_saligna_main_spread_sheet <- 
    
    # Read file
    read.csv("./euc_saligna_apr_droughtbox_main_spreadsheet.csv", header = TRUE,
             na.strings = "NA") %>% 
    
   # Remove columns
    select(!c(notes, number_of_parafilm_pieces, date, started_at,
              weight_paper_bag_plus_sample_scale_mg, 
              empty_paper_bag_plus_sample_scale_grams,
              bag_group)) %>% 
  
    # Assign median value to stems
    mutate(stem_dry_weight_mg = replace_na(stem_dry_weight_mg, median(stem_dry_weight_mg,
                                                                      na.rm=TRUE))) 

In [ ]:
glimpse(euc_saligna_main_spread_sheet)

In [ ]:
euc_saligna_dry_weights <-
  
    euc_saligna_main_spread_sheet %>% 
        
       mutate(
         leaf_dry_weight_g    = leaf_dry_weight_mg/1000, 
         sample_dry_weight_g =  ((leaf_dry_weight_mg/1000) +
                                         (stem_dry_weight_mg/1000) + 
                                         hook_weight_g + 
                                         prensa_weight_g), 
              # Add word to number 
              string_number = stringr::str_c("string_", string_number),
              .keep = "unused") %>% 
  
    na.omit()

## Merge dry weights data with droughtbox data

In [ ]:
euc_saligna_droughtbox_data_cleaned <- 

    euc_saligna_droughtbox_data_cleaned %>% 

        # Remove the word string inside the column string_number 
        mutate(set_temperature = temperature_measured, .keep = "unused") %>% 
            
        inner_join(., euc_saligna_dry_weights, 
                by = join_by(string_number, set_temperature, vpd_control)) %>% 
        
        select(-c(spcode, tree_id))

### Plot droughtbox data with dry weights

In [ ]:

#plot_no_vpd_control <- 
    euc_saligna_droughtbox_data_cleaned %>% 
        
        # Filter data
        filter(vpd_control == "no") %>% 

        ggplot(data = ., aes(x = time_seconds,
                             y = string_weight_grams,
                             colour = factor(string_number))) +
            geom_point() +
        
            theme_bw() +
            
            ggtitle("Data with no VPD control") +
        
            # Move legend at the bottom
            theme() +
        
            # Break plot into panels
            facet_grid(vars(set_temperature), vars(string_number)) +
        
            # At lines showing range between 0-3600 sec (1 hour)
            geom_vline(xintercept = 0,  linetype = "dotted") +
        
            # Set colors
            scale_color_manual(values = string_colors) +
        
            # Show dry-weights
            geom_hline(aes(yintercept = sample_dry_weight_g)) +

            # Rotate labels by 90 degrees
            theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
                  legend.position = "none")

In [ ]:

#plot_vpd_controlled <- 
    euc_saligna_droughtbox_data_cleaned %>% 
        
        # Filter data
        filter(vpd_control == "yes") %>% 

        ggplot(data = ., aes(x = time_seconds,
                             y = string_weight_grams,
                             colour = factor(string_number))) +
            geom_point() +
        
            theme_bw() +
            
            ggtitle("Data with VPD below 2kPa") +
        
            # Move legend at the bottom
            theme() +
        
            # Break plot into panels
            facet_grid(vars(set_temperature), vars(string_number)) +
        
            # At lines showing range between 0-3600 sec (1 hour)
            geom_vline(xintercept = 0,  linetype = "dotted") +
        
            # Set colors
            scale_color_manual(values = string_colors) +
        
            # Show dry-weights
            geom_hline(aes(yintercept = sample_dry_weight_g)) +
        
            # Rotate labels by 90 degrees
            theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
                  legend.position = "none")

# 4. Export data

In [ ]:
save(euc_saligna_droughtbox_data_cleaned, 
     file = "./euc_saligna_processed_data/euc_saligna_droughtbox_data_for_rwc.RData")